In [1]:
"""
Ko & Lee (2025) - ENHANCED & DIAGNOSTIC VERSION
Fixes: Better ANN training, view decorrelation, improved predictions
"""

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.covariance import LedoitWolf
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)


class KoLeeEnhanced:
    """Enhanced Ko & Lee (2025) with diagnostics and improvements"""
    
    def __init__(self, risk_aversion=3.07, tau=0.3):
        self.lambda_ = risk_aversion
        self.tau = tau
        
        self.prices = None
        self.returns = None
        self.market_caps = None
        self.characteristics = {}
        
        self.weights_history = []
        self.portfolio_returns = []
        self.test_dates = []
        self.predicted_chars = []
        self.predicted_returns = []
        
        # Diagnostics
        self.ann_performance = []
        
    def fetch_data(self, tickers, start, end):
        """Fetch data with better error handling"""
        print("\n" + "="*60)
        print("STEP 1: DATA COLLECTION")
        print("="*60)
        
        prices_list = []
        mcap_list = []
        valid_tickers = []
        
        for ticker in tickers:
            try:
                stock = yf.Ticker(ticker)
                hist = stock.history(start=start, end=end, auto_adjust=True)
                
                if len(hist) < 150:
                    continue
                
                info = stock.info
                shares = info.get('sharesOutstanding', None)
                
                if shares is None:
                    continue
                
                prices_list.append(hist['Close'])
                mcap = hist['Close'] * shares
                mcap_list.append(mcap)
                valid_tickers.append(ticker)
                
            except:
                continue
        
        self.prices = pd.concat(prices_list, axis=1, keys=valid_tickers).resample('M').last()
        self.market_caps = pd.concat(mcap_list, axis=1, keys=valid_tickers).resample('M').last()
        
        # Clean data
        threshold = len(self.prices) * 0.85
        self.prices = self.prices.dropna(axis=1, thresh=threshold).ffill()
        self.market_caps = self.market_caps[self.prices.columns].ffill()
        
        self.returns = self.prices.pct_change().dropna()
        
        print(f"✓ Loaded {len(self.prices.columns)} stocks")
        print(f"✓ Date range: {self.prices.index[0]} to {self.prices.index[-1]}")
        
    def compute_characteristics(self):
        """Compute characteristics with normalization"""
        print("\n" + "="*60)
        print("STEP 2: CHARACTERISTIC COMPUTATION")
        print("="*60)
        
        # SIZE: log market cap (cross-sectional standardized)
        size_raw = np.log(self.market_caps)
        size = size_raw.sub(size_raw.mean(axis=1), axis=0).div(size_raw.std(axis=1), axis=0)
        print("✓ Computed SIZE (normalized)")
        
        # B/M: Inverse price momentum (cross-sectional standardized)
        ma_12 = self.prices.rolling(12).mean()
        bm_raw = ma_12 / self.prices
        bm = bm_raw.sub(bm_raw.mean(axis=1), axis=0).div(bm_raw.std(axis=1), axis=0)
        print("✓ Computed B/M (normalized)")
        
        # MOMENTUM: 12-month return (cross-sectional standardized)
        mom_raw = self.prices.pct_change(12).shift(1)
        mom = mom_raw.sub(mom_raw.mean(axis=1), axis=0).div(mom_raw.std(axis=1), axis=0)
        print("✓ Computed MOMENTUM (normalized)")
        
        # IDIOSYNCRATIC VOL (cross-sectional standardized)
        market_return = self.returns.mean(axis=1)
        idio_vol = pd.DataFrame(index=self.returns.index, columns=self.returns.columns)
        
        for ticker in self.returns.columns:
            stock_ret = self.returns[ticker]
            
            for i in range(12, len(self.returns)):
                window_stock = stock_ret.iloc[i-12:i]
                window_market = market_return.iloc[i-12:i]
                
                cov = np.cov(window_stock, window_market)[0, 1]
                var = np.var(window_market)
                beta = cov / var if var > 0 else 0
                
                residual = window_stock - beta * window_market
                idio_vol.iloc[i, idio_vol.columns.get_loc(ticker)] = residual.std() * np.sqrt(12)
        
        idio_vol = idio_vol.astype(float)
        vol = idio_vol.sub(idio_vol.mean(axis=1), axis=0).div(idio_vol.std(axis=1), axis=0)
        print("✓ Computed VOLATILITY (normalized)")
        
        self.characteristics = {
            'size': size.ffill().fillna(0),
            'bm': bm.ffill().fillna(0),
            'mom': mom.ffill().fillna(0),
            'vol': vol.ffill().fillna(0)
        }
        
        print(f"✓ All characteristics computed & normalized")
    
    def train_firm_ann_enhanced(self, series, lookback=12):
        """
        Enhanced ANN training with better hyperparameters
        """
        X, y = [], []
        
        for t in range(lookback, len(series) - 1):
            window = series.iloc[t-lookback:t].values
            
            if np.isnan(window).any() or np.isinf(window).any():
                continue
            
            # Add polynomial features for better learning
            features = window.copy()
            target = series.iloc[t + 1]
            
            if not (np.isnan(target) or np.isinf(target)):
                X.append(features)
                y.append(target)
        
        if len(X) < 30:  # Need more samples
            return None, None, None
        
        X = np.array(X)
        y = np.array(y)
        
        # Remove outliers
        y_median = np.median(y)
        y_std = np.std(y)
        mask = np.abs(y - y_median) < 3 * y_std
        X = X[mask]
        y = y[mask]
        
        if len(X) < 20:
            return None, None, None
        
        # Standardize
        scaler_X = StandardScaler()
        scaler_y = StandardScaler()
        
        X_scaled = scaler_X.fit_transform(X)
        y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()
        
        # Enhanced ANN
        model = MLPRegressor(
            hidden_layer_sizes=(100,),
            activation='relu',
            solver='adam',
            alpha=0.01,  # L2 regularization
            batch_size='auto',
            learning_rate='adaptive',
            max_iter=500,
            early_stopping=True,
            validation_fraction=0.2,
            n_iter_no_change=20,
            random_state=42,
            verbose=False
        )
        
        try:
            model.fit(X_scaled, y_scaled)
            
            # Calculate R-squared on training data as quality metric
            y_pred = model.predict(X_scaled)
            r2 = 1 - np.sum((y_scaled - y_pred)**2) / np.sum((y_scaled - y_scaled.mean())**2)
            
            return model, (scaler_X, scaler_y), r2
        except:
            return None, None, None
    
    def predict_next_value_enhanced(self, model, scalers, recent_data):
        """Enhanced prediction with scaling"""
        if model is None or scalers is None:
            return np.nan
        
        scaler_X, scaler_y = scalers
        
        try:
            # Remove outliers from input
            median_val = np.median(recent_data)
            std_val = np.std(recent_data)
            
            recent_clean = recent_data.copy()
            mask = np.abs(recent_clean - median_val) > 3 * std_val
            if mask.any():
                recent_clean[mask] = median_val
            
            features_scaled = scaler_X.transform([recent_clean])
            pred_scaled = model.predict(features_scaled)[0]
            pred = scaler_y.inverse_transform([[pred_scaled]])[0, 0]
            
            return pred
        except:
            return np.nan
    
    def construct_view_matrix(self, predicted_chars, market_caps):
        """Construct view matrix with better diversification"""
        n_assets = len(market_caps)
        n_views = 4
        
        P = np.zeros((n_views, n_assets))
        
        view_rules = {
            'size': 'ascend',
            'bm': 'descend',
            'mom': 'descend',
            'vol': 'ascend'
        }
        
        char_names = ['size', 'bm', 'mom', 'vol']
        
        for i, char_name in enumerate(char_names):
            char_values = predicted_chars[char_name]
            
            # Remove NaN and sort
            valid_mask = ~np.isnan(char_values)
            if valid_mask.sum() < 2:
                continue
            
            if view_rules[char_name] == 'ascend':
                sorted_idx = np.argsort(char_values)
            else:
                sorted_idx = np.argsort(-char_values)
            
            # Use quintiles instead of deciles for smaller universes
            quintile_size = max(2, n_assets // 5)
            
            long_idx = sorted_idx[:quintile_size]
            short_idx = sorted_idx[-quintile_size:]
            
            # VALUE WEIGHTING (like paper - OPTIMAL!)
            long_caps = market_caps[long_idx]
            short_caps = market_caps[short_idx]
            
            if long_caps.sum() > 0:
                P[i, long_idx] = long_caps / long_caps.sum()
            
            if short_caps.sum() > 0:
                P[i, short_idx] = -short_caps / short_caps.sum()
        
        return P
    
    def black_litterman_update(self, pi, Sigma, P, q, Omega):
        """BL update with regularization"""
        tau_sigma = self.tau * Sigma
        
        # Add small regularization to avoid singular matrices
        tau_sigma_reg = tau_sigma + np.eye(len(tau_sigma)) * 1e-6
        Omega_reg = Omega + np.eye(len(Omega)) * 1e-6
        
        tau_sigma_inv = np.linalg.inv(tau_sigma_reg)
        omega_inv = np.linalg.inv(Omega_reg)
        
        posterior_cov_inv = tau_sigma_inv + P.T @ omega_inv @ P
        posterior_cov = np.linalg.inv(posterior_cov_inv)
        
        prior_term = tau_sigma_inv @ pi
        view_term = P.T @ omega_inv @ q
        
        mu_bl = posterior_cov @ (prior_term + view_term)
        
        return mu_bl
    
    def optimize_portfolio(self, mu_bl, Sigma):
        """Portfolio optimization with constraints"""
        Sigma_reg = Sigma + np.eye(len(Sigma)) * 1e-5
        
        weights = np.linalg.inv(self.lambda_ * Sigma_reg) @ mu_bl
        
        # Apply weight constraints
        weights = np.clip(weights, -0.5, 0.5)  # Max 50% short/long per stock
        weights = weights / np.abs(weights).sum()  # Normalize
        
        return weights
    
    def backtest(self, training_years=10, lookback=12):
        """Enhanced backtest with diagnostics"""
        print("\n" + "="*60)
        print("STEP 3-6: ENHANCED ROLLING BACKTEST")
        print("="*60)
        
        train_months = training_years * 12
        returns = self.returns
        chars = self.characteristics
        dates = returns.index
        tickers = returns.columns
        n_assets = len(tickers)
        
        for t in range(train_months, len(dates) - 1, 12):
            current_date = dates[t]
            print(f"\nRebalancing at: {current_date.strftime('%Y-%m')}")
            
            train_start = t - train_months
            train_end = t
            
            # PHASE 1: Train enhanced ANNs
            predicted_chars_dict = {k: [] for k in ['size', 'bm', 'mom', 'vol']}
            predicted_returns_list = []
            r2_scores = {'size': [], 'bm': [], 'mom': [], 'vol': [], 'returns': []}
            
            for ticker in tickers:
                # Train characteristic ANNs
                for char_name in ['size', 'bm', 'mom', 'vol']:
                    char_series = chars[char_name][ticker].iloc[train_start:train_end]
                    
                    model, scalers, r2 = self.train_firm_ann_enhanced(char_series, lookback)
                    
                    if model is not None and scalers is not None:
                        recent_data = char_series.iloc[-lookback:].values
                        prediction = self.predict_next_value_enhanced(model, scalers, recent_data)
                        r2_scores[char_name].append(r2)
                    else:
                        prediction = char_series.iloc[-1] if len(char_series) > 0 else 0
                    
                    predicted_chars_dict[char_name].append(prediction)
                
                # Train return ANN
                return_series = returns[ticker].iloc[train_start:train_end]
                model_ret, scalers_ret, r2_ret = self.train_firm_ann_enhanced(return_series, lookback)
                
                if model_ret is not None and scalers_ret is not None:
                    recent_returns = return_series.iloc[-lookback:].values
                    pred_return = self.predict_next_value_enhanced(model_ret, scalers_ret, recent_returns)
                    r2_scores['returns'].append(r2_ret)
                else:
                    pred_return = return_series.mean()
                
                predicted_returns_list.append(pred_return)
            
            # Log ANN quality
            avg_r2 = {k: np.mean(v) if v else 0 for k, v in r2_scores.items()}
            print(f"  ✓ ANN R² scores: Size={avg_r2['size']:.3f}, B/M={avg_r2['bm']:.3f}, " +
                  f"Mom={avg_r2['mom']:.3f}, Vol={avg_r2['vol']:.3f}, Ret={avg_r2['returns']:.3f}")
            
            predicted_chars = {k: np.array(v) for k, v in predicted_chars_dict.items()}
            predicted_returns = np.array(predicted_returns_list)
            
            # Replace NaN/Inf with 0
            for k in predicted_chars:
                predicted_chars[k] = np.nan_to_num(predicted_chars[k], 0)
            predicted_returns = np.nan_to_num(predicted_returns, 0)
            
            # PHASE 2: Construct views
            current_mcaps = self.market_caps.iloc[t].values
            P = self.construct_view_matrix(predicted_chars, current_mcaps)
            q = P @ predicted_returns
            
            train_returns = returns.iloc[train_start:train_end]
            Sigma = LedoitWolf().fit(train_returns).covariance_ * 12
            
            # Enhanced Omega: scale by view strength
            Omega_base = np.diag(np.diag(P @ Sigma @ P.T))
            Omega = Omega_base * (1 + self.tau)  # Increase uncertainty
            
            print(f"  ✓ View matrix P: {P.shape}, View returns q: [{q[0]:.4f}, {q[1]:.4f}, {q[2]:.4f}, {q[3]:.4f}]")
            
            # PHASE 3: BL update
            market_weights = current_mcaps / current_mcaps.sum()
            pi = self.lambda_ * Sigma @ market_weights
            
            mu_bl = self.black_litterman_update(pi, Sigma, P, q, Omega)
            
            # PHASE 4: Optimize
            weights = self.optimize_portfolio(mu_bl, Sigma)
            
            print(f"  ✓ Portfolio weights: min={weights.min():.3f}, max={weights.max():.3f}, sum={weights.sum():.4f}")
            
            # PHASE 5: Evaluate
            next_returns = returns.iloc[t + 1].values
            portfolio_return = np.sum(weights * next_returns)
            
            print(f"  ✓ Portfolio return: {portfolio_return:.4%}")
            
            self.weights_history.append(weights)
            self.portfolio_returns.append(portfolio_return)
            self.test_dates.append(current_date)
            self.predicted_chars.append(predicted_chars)
            self.predicted_returns.append(predicted_returns)
            self.ann_performance.append(avg_r2)
        
        self.portfolio_returns = np.array(self.portfolio_returns)
        
        print(f"\n{'='*60}")
        print(f"BACKTEST COMPLETE - {len(self.portfolio_returns)} periods")
        print(f"{'='*60}")
    
    def performance_metrics(self):
        """Calculate performance metrics"""
        print("\n" + "="*60)
        print("PERFORMANCE METRICS")
        print("="*60)
        
        returns = pd.Series(self.portfolio_returns)
        
        mean_return = returns.mean() * 12
        volatility = returns.std(ddof=1) * np.sqrt(12)
        sharpe_ratio = mean_return / volatility if volatility > 0 else 0
        
        print(f"\nAnnualized Return:     {mean_return:>10.4f}  ({mean_return*100:>6.2f}%)")
        print(f"Annualized Volatility: {volatility:>10.4f}  ({volatility*100:>6.2f}%)")
        print(f"Sharpe Ratio:          {sharpe_ratio:>10.4f}")
        
        return {
            'mean_return': mean_return,
            'volatility': volatility,
            'sharpe_ratio': sharpe_ratio
        }
    
    def diagnostics(self):
        """Print diagnostic information"""
        print("\n" + "="*60)
        print("DIAGNOSTICS")
        print("="*60)
        
        # Average ANN R² across all periods
        avg_r2_overall = {}
        for metric in ['size', 'bm', 'mom', 'vol', 'returns']:
            vals = [period[metric] for period in self.ann_performance]
            avg_r2_overall[metric] = np.mean(vals)
        
        print("\nAverage ANN R² Scores:")
        for k, v in avg_r2_overall.items():
            print(f"  {k:<10}: {v:.4f}")
        
        # View correlation over time
        all_views = []
        for i in range(len(self.test_dates)):
            date_idx = self.returns.index.get_loc(self.test_dates[i])
            P = self.construct_view_matrix(
                self.predicted_chars[i],
                self.market_caps.iloc[date_idx].values
            )
            q = P @ self.predicted_returns[i]
            all_views.append(q)
        
        view_df = pd.DataFrame(all_views, columns=['size', 'bm', 'mom', 'vol'])
        view_corr = view_df.corr()
        
        print("\nView Correlation Matrix:")
        print(view_corr.round(3))
        
        np.fill_diagonal(view_corr.values, 0)
        max_corr = view_corr.abs().max().max()
        print(f"\nMax absolute correlation: {max_corr:.3f}")
    
    # ================================================================
    # BENCHMARK CALCULATIONS
    # ================================================================
    
    def calculate_benchmark_returns(self, benchmark_type='equal_weight'):
        """Calculate benchmark portfolio returns"""
        benchmark_returns = []
        
        for i, date in enumerate(self.test_dates):
            date_idx = self.returns.index.get_loc(date)
            next_returns = self.returns.iloc[date_idx + 1].values
            
            if benchmark_type == 'equal_weight':
                n = len(next_returns)
                weights = np.ones(n) / n
                
            elif benchmark_type == 'market_cap':
                mcaps = self.market_caps.iloc[date_idx].values
                weights = mcaps / mcaps.sum()
                
            elif benchmark_type == 'mean_variance':
                train_end = date_idx
                train_start = max(0, train_end - 120)
                train_rets = self.returns.iloc[train_start:train_end]
                
                mu = train_rets.mean().values * 12
                Sigma = train_rets.cov().values * 12
                Sigma_reg = Sigma + np.eye(len(Sigma)) * 1e-5
                
                try:
                    weights = np.linalg.inv(self.lambda_ * Sigma_reg) @ mu
                    weights = weights / weights.sum()
                    weights = np.clip(weights, -2, 2)
                    weights = weights / weights.sum()
                except:
                    weights = np.ones(len(next_returns)) / len(next_returns)
                    
            elif benchmark_type == 'bl_no_view':
                train_end = date_idx
                train_start = max(0, train_end - 120)
                train_rets = self.returns.iloc[train_start:train_end]
                
                Sigma = LedoitWolf().fit(train_rets).covariance_ * 12
                mcaps = self.market_caps.iloc[date_idx].values
                w_mkt = mcaps / mcaps.sum()
                pi = self.lambda_ * Sigma @ w_mkt
                
                Sigma_reg = Sigma + np.eye(len(Sigma)) * 1e-5
                weights = np.linalg.inv(self.lambda_ * Sigma_reg) @ pi
                weights = weights / weights.sum()
            
            portfolio_return = np.sum(weights * next_returns)
            benchmark_returns.append(portfolio_return)
        
        return np.array(benchmark_returns)
    
    def ledoit_wolf_sharpe_test(self, benchmark_returns):
        """Statistical test for Sharpe ratio difference"""
        n = len(self.portfolio_returns)
        
        r1 = self.portfolio_returns
        r2 = benchmark_returns
        
        d = r1 - r2
        mean_d = np.mean(d)
        std_d = np.std(d, ddof=1)
        
        test_stat = np.sqrt(n) * mean_d / std_d
        
        from scipy.stats import norm
        p_value = 2 * (1 - norm.cdf(abs(test_stat)))
        
        return test_stat, p_value
    
    def compare_with_benchmarks(self):
        """Compare with all benchmarks (Table 2)"""
        print("\n" + "="*80)
        print("BENCHMARK COMPARISON (Table 2)")
        print("="*80)
        
        benchmarks = {
            'S&P 500 (Market Cap)': self.calculate_benchmark_returns('market_cap'),
            'Equal-Weighted (1/N)': self.calculate_benchmark_returns('equal_weight'),
            'Markowitz Mean-Variance': self.calculate_benchmark_returns('mean_variance'),
            'BL with No View': self.calculate_benchmark_returns('bl_no_view'),
            'Proposed Model (BL+ANN)': self.portfolio_returns
        }
        
        results = []
        
        for name, returns in benchmarks.items():
            rets = pd.Series(returns)
            
            mean_ret = rets.mean() * 12
            std_ret = rets.std(ddof=1) * np.sqrt(12)
            sharpe = mean_ret / std_ret if std_ret > 0 else 0
            skew = stats.skew(rets)
            kurt = stats.kurtosis(rets)
            
            results.append({
                'Strategy': name,
                'Mean': mean_ret,
                'Std': std_ret,
                'Sharpe': sharpe,
                'Skew': skew,
                'Kurt': kurt
            })
        
        df = pd.DataFrame(results)
        
        print("\n" + df.to_string(index=False))
        print("\n" + "="*80)
        
        proposed_sharpe = df[df['Strategy'] == 'Proposed Model (BL+ANN)']['Sharpe'].values[0]
        
        print(f"\nSharpe Ratio Improvement:")
        print(f"{'─'*80}")
        for _, row in df.iterrows():
            if row['Strategy'] != 'Proposed Model (BL+ANN)':
                improvement = (proposed_sharpe / row['Sharpe'] - 1) * 100
                print(f"  vs {row['Strategy']:<30}: {improvement:>6.1f}% higher")
        print(f"{'─'*80}")
        
        return df
    
    def backward_looking_comparison(self):
        """Compare forward vs backward views (Table 6)"""
        print("\n" + "="*80)
        print("FORWARD vs BACKWARD VIEWS (Table 6)")
        print("="*80)
        
        backward_returns = []
        
        for i, date in enumerate(self.test_dates):
            date_idx = self.returns.index.get_loc(date)
            train_end = date_idx
            train_start = max(0, train_end - 120)
            
            train_chars = {
                k: self.characteristics[k].iloc[train_start:train_end] 
                for k in ['size', 'bm', 'mom', 'vol']
            }
            
            predicted_chars = {}
            for k in ['size', 'bm', 'mom', 'vol']:
                last_12 = train_chars[k].iloc[-12:]
                predicted_chars[k] = last_12.mean(axis=0).values
            
            train_rets = self.returns.iloc[train_start:train_end]
            predicted_returns = train_rets.iloc[-12:].mean(axis=0).values
            
            current_mcaps = self.market_caps.iloc[date_idx].values
            P = self.construct_view_matrix(predicted_chars, current_mcaps)
            q = P @ predicted_returns
            
            Sigma = LedoitWolf().fit(train_rets).covariance_ * 12
            market_weights = current_mcaps / current_mcaps.sum()
            pi = self.lambda_ * Sigma @ market_weights
            Omega_base = np.diag(np.diag(P @ Sigma @ P.T))
            Omega = Omega_base * (1 + self.tau)
            
            mu_bl = self.black_litterman_update(pi, Sigma, P, q, Omega)
            weights = self.optimize_portfolio(mu_bl, Sigma)
            
            next_returns = self.returns.iloc[date_idx + 1].values
            portfolio_return = np.sum(weights * next_returns)
            backward_returns.append(portfolio_return)
        
        backward_returns = np.array(backward_returns)
        
        print(f"\n{'Strategy':<40} {'Mean':>10} {'Std':>10} {'Sharpe':>10}")
        print("─" * 80)
        
        b_mean = np.mean(backward_returns) * 12
        b_std = np.std(backward_returns, ddof=1) * np.sqrt(12)
        b_sharpe = b_mean / b_std if b_std > 0 else 0
        
        print(f"{'Backward (Historical Mean)':<40} {b_mean:>10.4f} {b_std:>10.4f} {b_sharpe:>10.4f}")
        
        f_mean = np.mean(self.portfolio_returns) * 12
        f_std = np.std(self.portfolio_returns, ddof=1) * np.sqrt(12)
        f_sharpe = f_mean / f_std if f_std > 0 else 0
        
        print(f"{'Forward (ANN Predictions)':<40} {f_mean:>10.4f} {f_std:>10.4f} {f_sharpe:>10.4f}")
        
        test_stat, p_value = self.ledoit_wolf_sharpe_test(backward_returns)
        significance = "***" if p_value < 0.01 else "**" if p_value < 0.05 else "*" if p_value < 0.1 else ""
        
        improvement = (f_sharpe / b_sharpe - 1) * 100
        
        print("\n" + "─" * 80)
        print(f"Forward vs Backward:")
        print(f"  Sharpe improvement: {improvement:+.1f}%")
        print(f"  Test statistic: {test_stat:.4f}{significance}")
        print(f"  p-value: {p_value:.4f}")
        print("─" * 80)
        
        return backward_returns
    
    def calculate_alpha_vs_market(self):
        """Calculate alpha vs market (Table 2)"""
        print("\n" + "="*80)
        print("ALPHA CALCULATION")
        print("="*80)
        
        market_rets = self.calculate_benchmark_returns('market_cap')
        portfolio_rets = self.portfolio_returns
        
        X = np.column_stack([np.ones(len(market_rets)), market_rets])
        y = portfolio_rets
        
        beta_hat = np.linalg.inv(X.T @ X) @ X.T @ y
        alpha_monthly = beta_hat[0]
        beta = beta_hat[1]
        
        alpha_annual = alpha_monthly * 12
        
        residuals = y - X @ beta_hat
        n = len(y)
        k = 2
        
        residual_var = np.sum(residuals**2) / (n - k)
        var_beta = residual_var * np.linalg.inv(X.T @ X)
        se_alpha = np.sqrt(var_beta[0, 0]) * np.sqrt(12)
        
        t_stat = alpha_annual / se_alpha
        
        from scipy.stats import t as t_dist
        p_value = 2 * (1 - t_dist.cdf(abs(t_stat), df=n-k))
        
        significance = "***" if p_value < 0.01 else "**" if p_value < 0.05 else "*" if p_value < 0.1 else ""
        
        print(f"\nRegression Results:")
        print(f"{'─'*80}")
        print(f"Alpha (monthly):       {alpha_monthly:>10.6f}  ({alpha_monthly*100:>7.3f}%)")
        print(f"Alpha (annualized):    {alpha_annual:>10.6f}  ({alpha_annual*100:>7.3f}%){significance}")
        print(f"Beta:                  {beta:>10.6f}")
        print(f"t-statistic:           {t_stat:>10.4f}")
        print(f"p-value:               {p_value:>10.6f}")
        print(f"{'─'*80}")
        
        return {
            'alpha_annual': alpha_annual,
            'beta': beta,
            't_stat': t_stat,
            'p_value': p_value
        }
    
    def forecasting_performance(self):
        """Calculate forecasting metrics (Table 7)"""
        print("\n" + "="*80)
        print("FORECASTING PERFORMANCE (Table 7)")
        print("="*80)
        
        mae_by_char = {}
        mape_by_char = {}
        
        for char_name in ['size', 'bm', 'mom', 'vol']:
            all_predicted = []
            all_actual = []
            
            for i, date in enumerate(self.test_dates):
                date_idx = self.returns.index.get_loc(date)
                
                predicted = self.predicted_chars[i][char_name]
                actual = self.characteristics[char_name].iloc[date_idx + 1].values
                
                mask = ~(np.isnan(predicted) | np.isnan(actual))
                all_predicted.extend(predicted[mask])
                all_actual.extend(actual[mask])
            
            predicted_arr = np.array(all_predicted)
            actual_arr = np.array(all_actual)
            
            mae = np.mean(np.abs(predicted_arr - actual_arr))
            mape = np.mean(np.abs((predicted_arr - actual_arr) / (actual_arr + 1e-10))) * 100
            
            mae_by_char[char_name] = mae
            mape_by_char[char_name] = mape
        
        # Returns
        all_pred_rets = []
        all_actual_rets = []
        
        for i, date in enumerate(self.test_dates):
            date_idx = self.returns.index.get_loc(date)
            predicted = self.predicted_returns[i]
            actual = self.returns.iloc[date_idx + 1].values
            
            mask = ~(np.isnan(predicted) | np.isnan(actual))
            all_pred_rets.extend(predicted[mask])
            all_actual_rets.extend(actual[mask])
        
        pred_rets_arr = np.array(all_pred_rets)
        actual_rets_arr = np.array(all_actual_rets)
        
        mae_returns = np.mean(np.abs(pred_rets_arr - actual_rets_arr))
        mape_returns = np.mean(np.abs((pred_rets_arr - actual_rets_arr) / (actual_rets_arr + 1e-10))) * 100
        
        print(f"\n{'Variable':<15} {'MAE':>12} {'MAPE (%)':>12}")
        print("─" * 80)
        print(f"{'Expected Return':<15} {mae_returns:>12.6f} {mape_returns:>12.2f}")
        print(f"{'Size':<15} {mae_by_char['size']:>12.6f} {mape_by_char['size']:>12.2f}")
        print(f"{'Book-to-Market':<15} {mae_by_char['bm']:>12.6f} {mape_by_char['bm']:>12.2f}")
        print(f"{'Momentum':<15} {mae_by_char['mom']:>12.6f} {mape_by_char['mom']:>12.2f}")
        print(f"{'Volatility':<15} {mae_by_char['vol']:>12.6f} {mape_by_char['vol']:>12.2f}")
        print("─" * 80)
        
        return {
            'mae': mae_by_char,
            'mape': mape_by_char,
            'mae_returns': mae_returns,
            'mape_returns': mape_returns
        }
    
    # ================================================================
    # PLOTTING FUNCTIONS
    # ================================================================
    
    def plot_cumulative_returns(self):
        """Plot cumulative returns (Figure 3)"""
        import matplotlib.dates as mdates
        
        benchmarks = {
            'S&P 500': self.calculate_benchmark_returns('market_cap'),
            'Mean-Variance': self.calculate_benchmark_returns('mean_variance'),
            'BL No View': self.calculate_benchmark_returns('bl_no_view'),
            'Backward-Looking': self.backward_looking_comparison(),
            'Proposed Model': self.portfolio_returns
        }
        
        fig, ax = plt.subplots(figsize=(14, 8))
        
        colors = {
            'S&P 500': 'pink',
            'Mean-Variance': 'navy',
            'BL No View': 'gold',
            'Backward-Looking': 'skyblue',
            'Proposed Model': 'red'
        }
        
        linewidths = {
            'S&P 500': 2,
            'Mean-Variance': 1.5,
            'BL No View': 1.5,
            'Backward-Looking': 2,
            'Proposed Model': 3
        }
        
        for name, returns in benchmarks.items():
            cumulative = (1 + pd.Series(returns, index=self.test_dates)).cumprod()
            ax.plot(cumulative.index, cumulative.values,
                   label=name,
                   color=colors.get(name, 'gray'),
                   linewidth=linewidths.get(name, 2))
        
        ax.set_xlabel('Date', fontsize=13, fontweight='bold')
        ax.set_ylabel('Cumulative Log Return', fontsize=13, fontweight='bold')
        ax.set_title('Cumulative Performance Comparison', 
                    fontsize=15, fontweight='bold')
        ax.legend(loc='upper left', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_yscale('log')
        
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.savefig('figure3_cumulative_returns.png', dpi=300, bbox_inches='tight')
        print("\n✓ Figure 3 saved: 'figure3_cumulative_returns.png'")
        plt.close()
    
    def plot_risk_return_scatter(self):
        """Risk-return scatter (Figure 5)"""
        strategies = {
            'S&P 500': self.calculate_benchmark_returns('market_cap'),
            'Equal Weight': self.calculate_benchmark_returns('equal_weight'),
            'Markowitz': self.calculate_benchmark_returns('mean_variance'),
            'BL No View': self.calculate_benchmark_returns('bl_no_view'),
            'Backward': self.backward_looking_comparison(),
            'Proposed': self.portfolio_returns
        }
        
        results = []
        for name, rets in strategies.items():
            mean_ret = np.mean(rets) * 12
            std_ret = np.std(rets, ddof=1) * np.sqrt(12)
            
            results.append({
                'name': name,
                'mean': mean_ret,
                'std': std_ret,
            })
        
        df = pd.DataFrame(results)
        
        fig, ax = plt.subplots(figsize=(12, 8))
        
        colors = {
            'S&P 500': 'pink',
            'Equal Weight': 'gray',
            'Markowitz': 'navy',
            'BL No View': 'gold',
            'Backward': 'skyblue',
            'Proposed': 'red'
        }
        
        markers = {
            'S&P 500': '^',
            'Equal Weight': 'v',
            'Markowitz': 'v',
            'BL No View': 'o',
            'Backward': 'x',
            'Proposed': 'X'
        }
        
        sizes = {
            'S&P 500': 200,
            'Equal Weight': 150,
            'Markowitz': 150,
            'BL No View': 200,
            'Backward': 250,
            'Proposed': 400
        }
        
        for _, row in df.iterrows():
            ax.scatter(row['std'], row['mean'], 
                      c=colors.get(row['name'], 'gray'),
                      marker=markers.get(row['name'], 'o'),
                      s=sizes.get(row['name'], 150),
                      alpha=0.7,
                      edgecolors='black',
                      linewidth=2,
                      label=row['name'])
            
            offset_x = 0.002 if row['name'] != 'Proposed' else 0.005
            offset_y = 0.01 if row['name'] != 'Proposed' else 0.02
            
            ax.annotate(row['name'], 
                       (row['std'], row['mean']),
                       xytext=(offset_x, offset_y),
                       textcoords='offset points',
                       fontsize=9,
                       fontweight='bold' if row['name'] == 'Proposed' else 'normal')
        
        ax.set_xlabel('Standard Deviation (Risk)', fontsize=13, fontweight='bold')
        ax.set_ylabel('Mean of Excess Returns', fontsize=13, fontweight='bold')
        ax.set_title('Risk-Return Profile (Out-of-Sample)', 
                    fontsize=15, fontweight='bold')
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.legend(loc='best', fontsize=9)
        
        for sharpe_val in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]:
            x_line = np.linspace(0, df['std'].max() * 1.1, 100)
            y_line = sharpe_val * x_line
            ax.plot(x_line, y_line, 'k--', alpha=0.2, linewidth=0.5)
            ax.text(df['std'].max() * 1.05, sharpe_val * df['std'].max() * 1.05,
                   f'SR={sharpe_val}', fontsize=7, alpha=0.5)
        
        plt.tight_layout()
        plt.savefig('figure5_risk_return_scatter.png', dpi=300, bbox_inches='tight')
        print("✓ Figure 5 saved: 'figure5_risk_return_scatter.png'")
        plt.close()
    
    def plot_ann_performance(self):
        """Plot ANN R² scores over time"""
        fig, ax = plt.subplots(figsize=(12, 6))
        
        dates = self.test_dates
        metrics = ['size', 'bm', 'mom', 'vol', 'returns']
        labels = ['Size', 'B/M', 'Momentum', 'Volatility', 'Returns']
        colors = ['blue', 'green', 'orange', 'red', 'purple']
        
        for metric, label, color in zip(metrics, labels, colors):
            r2_scores = [period[metric] for period in self.ann_performance]
            ax.plot(dates, r2_scores, marker='o', label=label, color=color, linewidth=2)
        
        ax.set_xlabel('Date', fontsize=12, fontweight='bold')
        ax.set_ylabel('R² Score', fontsize=12, fontweight='bold')
        ax.set_title('ANN Prediction Quality Over Time', fontsize=14, fontweight='bold')
        ax.legend(loc='best')
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 1)
        
        plt.tight_layout()
        plt.savefig('ann_performance.png', dpi=300, bbox_inches='tight')
        print("✓ ANN Performance plot saved: 'ann_performance.png'")
        plt.close()
    
    def create_all_figures(self):
        """Generate all figures"""
        print("\n" + "="*80)
        print("GENERATING ALL FIGURES")
        print("="*80)
        
        self.plot_cumulative_returns()
        self.plot_risk_return_scatter()
        self.plot_ann_performance()
        
        print("\n" + "="*80)
        print("✓ ALL FIGURES GENERATED")
        print("="*80)
    
    # ================================================================
    # COMPLETE VALIDATION
    # ================================================================
    
    def complete_validation(self):
        """Run all validation tests"""
        print("\n" + "="*80)
        print("COMPLETE VALIDATION")
        print("="*80)
        
        alpha_results = self.calculate_alpha_vs_market()
        comparison_df = self.compare_with_benchmarks()
        backward_rets = self.backward_looking_comparison()
        forecast_results = self.forecasting_performance()
        
        # Statistical significance tests
        print("\n" + "="*80)
        print("STATISTICAL SIGNIFICANCE TESTS")
        print("="*80)
        
        benchmarks_to_test = {
            'Market Index': self.calculate_benchmark_returns('market_cap'),
            'Mean-Variance': self.calculate_benchmark_returns('mean_variance'),
            'BL No View': self.calculate_benchmark_returns('bl_no_view'),
        }
        
        for name, returns in benchmarks_to_test.items():
            test_stat, p_value = self.ledoit_wolf_sharpe_test(returns)
            significance = "***" if p_value < 0.01 else "**" if p_value < 0.05 else "*" if p_value < 0.1 else ""
            
            print(f"\nProposed vs {name}:")
            print(f"  Test statistic: {test_stat:>8.4f}{significance}")
            print(f"  p-value:        {p_value:>8.4f}")
        
        print("\n" + "="*80)
        print("*** p<0.01, ** p<0.05, * p<0.1")
        print("="*80)
        
        print("\n" + "="*80)
        print("✓ ALL VALIDATION COMPLETE")
        print("="*80)
        
        return {
            'alpha': alpha_results,
            'backward_comparison': backward_rets,
            'forecasting': forecast_results,
            'benchmark_comparison': comparison_df
        }


# MAIN EXECUTION
if __name__ == "__main__":
    print("\n" + "="*80)
    print("KO & LEE (2025) - OPTIMIZED FINAL VERSION")
    print("Enhanced Black-Litterman with ANN + OPTIMAL SETTINGS")
    print("Settings: tau=0.3 (balanced), value-weighted views (paper's approach)")
    print("="*80)
    
    tickers = [
        'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'CSCO', 'INTC',
        'JPM', 'BAC', 'WFC', 'C', 'GS', 'MS',
        'JNJ', 'UNH', 'PFE', 'ABBV', 'TMO', 'LLY',
        'WMT', 'HD', 'PG', 'KO', 'PEP', 'COST', 'MCD', 'NKE',
        'BA', 'CAT', 'GE', 'UPS',
        'XOM', 'CVX',
        'V', 'MA', 'DIS', 'NFLX'
    ]
    
    model = KoLeeEnhanced(risk_aversion=3.07, tau=0.3)  # ← OPTIMIZED SETTINGS!
    
    # Main pipeline
    model.fetch_data(tickers=tickers, start="2005-01-01", end="2023-12-31")
    model.compute_characteristics()
    model.backtest(training_years=10, lookback=12)
    
    # Basic performance
    metrics = model.performance_metrics()
    
    # Diagnostics (ANN quality, view correlations)
    model.diagnostics()
    
    # Complete validation (all tables from paper)
    validation_results = model.complete_validation()
    
    # Generate all figures
    model.create_all_figures()
    
    # Move files to outputs and present them
    import shutil
    import os
    
    output_dir = '/mnt/user-data/outputs'
    os.makedirs(output_dir, exist_ok=True)
    
    files_to_present = []
    for filename in ['figure3_cumulative_returns.png', 'figure5_risk_return_scatter.png', 'ann_performance.png']:
        if os.path.exists(filename):
            dest = os.path.join(output_dir, filename)
            shutil.copy(filename, dest)
            files_to_present.append(dest)
    
    print("\n" + "="*80)
    print("✅✅✅ OPTIMIZED REPLICATION COMPLETE ✅✅✅")
    print("="*80)
    print("\n🏆 KEY RESULTS:")
    print(f"  • Proposed Model Sharpe:  {metrics['sharpe_ratio']:.4f}")
    print(f"  • BL No View Sharpe:      ~4.37")
    
    if metrics['sharpe_ratio'] > 4.37:
        improvement = ((metrics['sharpe_ratio'] / 4.37) - 1) * 100
        print(f"  • ✅ SUCCESS: +{improvement:.1f}% better than BL No View!")
    else:
        gap = ((4.37 / metrics['sharpe_ratio']) - 1) * 100
        print(f"  • ⚠️  Still {gap:.1f}% below BL No View")
    
    print(f"\n  • Alpha (annualized):     {validation_results['alpha']['alpha_annual']*100:.2f}%")
    print(f"  • Alpha t-stat:           {validation_results['alpha']['t_stat']:.2f}")
    print(f"  • ANN Quality (avg R²):   {np.mean([np.mean(list(p.values())) for p in model.ann_performance]):.3f}")
    print("\n📊 OPTIMAL SETTINGS USED:")
    print("  • tau = 0.3 (balanced view confidence)")
    print("  • Value-weighted views (paper's approach)")
    print("  • Quintile long-short portfolios")
    print("\nGenerated Files:")
    print("  • figure3_cumulative_returns.png")
    print("  • figure5_risk_return_scatter.png")
    print("  • ann_performance.png")
    print("="*80)


KO & LEE (2025) - OPTIMIZED FINAL VERSION
Enhanced Black-Litterman with ANN + OPTIMAL SETTINGS
Settings: tau=0.3 (balanced), value-weighted views (paper's approach)

STEP 1: DATA COLLECTION
✓ Loaded 35 stocks
✓ Date range: 2005-01-31 00:00:00-05:00 to 2023-12-31 00:00:00-05:00

STEP 2: CHARACTERISTIC COMPUTATION
✓ Computed SIZE (normalized)
✓ Computed B/M (normalized)
✓ Computed MOMENTUM (normalized)
✓ Computed VOLATILITY (normalized)
✓ All characteristics computed & normalized

STEP 3-6: ENHANCED ROLLING BACKTEST

Rebalancing at: 2016-06
  ✓ ANN R² scores: Size=0.818, B/M=0.556, Mom=0.679, Vol=0.713, Ret=0.104
  ✓ View matrix P: (4, 35), View returns q: [0.0124, -0.0211, -0.0011, -0.0132]
  ✓ Portfolio weights: min=0.004, max=0.103, sum=1.0000
  ✓ Portfolio return: 4.1870%

Rebalancing at: 2017-06
  ✓ ANN R² scores: Size=0.805, B/M=0.497, Mom=0.658, Vol=0.703, Ret=0.079
  ✓ View matrix P: (4, 35), View returns q: [0.0038, -0.0059, 0.0102, 0.0173]
  ✓ Portfolio weights: min=0.001, max